In [6]:
import time
import urllib.request
import urllib.error
import numpy as np
import networkx as nx
import cvxpy as cp

# =========================
# 1. Online Benchmark Fetcher (GSET)
# =========================
def fetch_gset_graph(gset_id="G1"):
    """
    Downloads and parses a GSET benchmark graph from a public mirror.
    GSET graphs range from G1 to G81.
    """
    # A reliable public mirror for GSET text files
    url = f"https://raw.githubusercontent.com/0816keisuke/max-cut-problem-benchmark/master/Gset/{gset_id}.txt"
    print(f"Fetching {gset_id} from online mirror...")
    
    try:
        response = urllib.request.urlopen(url)
        lines = response.read().decode('utf-8').splitlines()
    except urllib.error.URLError as e:
        print(f"Failed to download {gset_id}: {e}")
        return None

    # First line of GSET is usually: "num_nodes num_edges"
    # Subsequent lines: "node1 node2 weight"
    first_line = lines[0].strip().split()
    n = int(first_line[0])
    
    G = nx.Graph()
    G.add_nodes_from(range(1, n + 1)) # GSET is usually 1-indexed
    
    for line in lines[1:]:
        if not line.strip(): continue
        u, v, w = map(int, line.split())
        G.add_edge(u, v, weight=w)
        
    G.name = f"GSET_{gset_id}"
    return G

# =========================
# 2. Comprehensive Graph Generator
# =========================
def generate_graphs(configs):
    """
    Takes a list of configuration dictionaries and generates/fetches graphs.
    """
    graphs = []
    
    for idx, config in enumerate(configs):
        g_type = config.get('type')
        n = config.get('n', 50)
        seed = config.get('seed', 42 + idx)
        
        if g_type == 'gset':
            G = fetch_gset_graph(config.get('id', 'G1'))
            if G is not None:
                graphs.append(G)
            continue
            
        elif g_type == 'erdos_renyi':
            G = nx.erdos_renyi_graph(n, config.get('p', 0.3), seed=seed)
        elif g_type == 'regular':
            G = nx.random_regular_graph(config.get('d', 3), n, seed=seed)
        elif g_type == 'barabasi_albert':
            G = nx.barabasi_albert_graph(n, config.get('m', 3), seed=seed)
        elif g_type == 'watts_strogatz':
            # Small-world network (good for clustering)
            G = nx.watts_strogatz_graph(n, k=config.get('k', 4), p=config.get('p', 0.1), seed=seed)
        elif g_type == 'grid_2d':
            dim = int(np.sqrt(n))
            G = nx.grid_2d_graph(dim, dim)
            # Relabel tuples to integers for CVXPY simplicity
            G = nx.convert_node_labels_to_integers(G) 
        elif g_type == 'complete':
            G = nx.complete_graph(n)
        elif g_type == 'tree':
            G = nx.random_tree(n, seed=seed)
        else:
            print(f"Skipping unknown type: {g_type}")
            continue

        G.name = f"{g_type.upper()}_n{n}_{idx}"
        
        # Ensure all synthetic graphs have a default weight of 1
        for u, v in G.edges():
            G[u][v]['weight'] = 1.0
            
        graphs.append(G)
        
    return graphs

# =========================
# 3. Solve SDP Relaxation
# =========================
def solve_sdp(G):
    n = G.number_of_nodes()
    X = cp.Variable((n, n), symmetric=True)
    constraints = [X >> 0, cp.diag(X) == 1]

    # Use the weighted graph Laplacian
    L = nx.laplacian_matrix(G, weight='weight').toarray()
    obj = cp.trace(L @ X) / 4

    prob = cp.Problem(cp.Maximize(obj), constraints)
    
    # If dealing with GSET graphs, SCS might hit iteration limits. 
    # You may need to pass kwargs like: prob.solve(solver=cp.SCS, max_iters=5000)
    prob.solve(solver=cp.SCS)

    return X.value, prob.value

# =========================
# 4. Factor SDP Solution
# =========================
def factorize_sdp(X):
    eigvals, eigvecs = np.linalg.eigh(X)
    eigvals = np.clip(eigvals, 0, None)
    return eigvecs @ np.diag(np.sqrt(eigvals))

# =========================
# 5. Hyperplane Rounding
# =========================
def gw_rounding(G, V, num_samples=2000):
    n, d = V.shape
    
    # Extract edges and weights
    edges = list(G.edges(data='weight', default=1.0))
    edge_idx = np.array([[u, v] for u, v, w in edges])
    
    # Handle the fact that GSET might be 1-indexed or non-sequential
    # We need a mapping from node name to row index in V
    nodes = list(G.nodes())
    node_to_idx = {node: i for i, node in enumerate(nodes)}
    
    mapped_edges = np.array([[node_to_idx[u], node_to_idx[v]] for u, v in edge_idx])
    weights = np.array([w for u, v, w in edges])

    R = np.random.randn(d, num_samples)
    R /= np.linalg.norm(R, axis=0)

    signs = np.sign(V @ R)
    signs[signs == 0] = 1 

    # Vectorized weighted cut calculation
    # Check if endpoints are in different partitions (True/False)
    cut_mask = signs[mapped_edges[:, 0]] != signs[mapped_edges[:, 1]]
    
    # Multiply by weights and sum along the edge axis
    cuts = np.sum(cut_mask * weights[:, np.newaxis], axis=0)

    best_idx = np.argmax(cuts)
    return cuts[best_idx], signs[:, best_idx]

# =========================
# 6. Full Pipeline
# =========================
def goemans_williamson(G, num_samples=2000):
    t0 = time.perf_counter()
    X, sdp_val = solve_sdp(G)
    sdp_time = time.perf_counter() - t0

    t1 = time.perf_counter()
    V = factorize_sdp(X)
    cut, partition = gw_rounding(G, V, num_samples)
    rounding_time = time.perf_counter() - t1

    approx_ratio = cut / sdp_val if sdp_val > 0 else 0

    return {
        "graph_name": getattr(G, "name", "Unnamed"),
        "nodes": G.number_of_nodes(),
        "edges": G.number_of_edges(),
        "sdp_val": sdp_val,
        "cut_val": cut,
        "approx_ratio": approx_ratio,
        "sdp_time": sdp_time,
        "rounding_time": rounding_time,
        "total_time": sdp_time + rounding_time
    }

# =========================
# RUN BATCH EXPERIMENTS
# =========================
if __name__ == "__main__":
    
    # Define a rich batch of graphs to test
    # Note: G1 is 800 nodes. It might take 1-3 minutes to solve depending on your CPU.
    experiment_configs = [
        {'type': 'erdos_renyi', 'n': 100, 'p': 0.1},
        {'type': 'watts_strogatz', 'n': 100, 'k': 6, 'p': 0.1}, # Small world
        {'type': 'grid_2d', 'n': 100},                          # 10x10 Grid
        {'type': 'gset', 'id': 'G1'},                           # 800 nodes, 19k edges
    ]
    
    print("Generating/Fetching graphs...")
    graphs = generate_graphs(experiment_configs)
    
    all_results = []
    
    print("\nStarting Benchmark Pipeline...\n" + "="*60)
    for G in graphs:
        print(f"Solving {G.name} (|V|={G.number_of_nodes()}, |E|={G.number_of_edges()})...")
        res = goemans_williamson(G, num_samples=3000)
        all_results.append(res)
        
        print(f"  ├─ SDP Upper Bound: {res['sdp_val']:.2f}")
        print(f"  ├─ Actual Cut:      {res['cut_val']}")
        print(f"  ├─ Approx Ratio:    {res['approx_ratio']:.4f}")
        print(f"  └─ Total Time:      {res['total_time']:.3f}s (SDP: {res['sdp_time']:.3f}s, Rnd: {res['rounding_time']:.3f}s)\n")
        print("-" * 60)

Generating/Fetching graphs...
Fetching G1 from online mirror...
Failed to download G1: HTTP Error 404: Not Found

Starting Benchmark Pipeline...
Solving ERDOS_RENYI_n100_0 (|V|=100, |E|=474)...
  ├─ SDP Upper Bound: 358.66
  ├─ Actual Cut:      341.0
  ├─ Approx Ratio:    0.9508
  └─ Total Time:      0.258s (SDP: 0.240s, Rnd: 0.018s)

------------------------------------------------------------
Solving WATTS_STROGATZ_n100_1 (|V|=100, |E|=300)...
  ├─ SDP Upper Bound: 224.56
  ├─ Actual Cut:      214.0
  ├─ Approx Ratio:    0.9530
  └─ Total Time:      1.652s (SDP: 1.640s, Rnd: 0.012s)

------------------------------------------------------------
Solving GRID_2D_n100_2 (|V|=100, |E|=180)...
  ├─ SDP Upper Bound: 180.00
  ├─ Actual Cut:      180.0
  ├─ Approx Ratio:    1.0000
  └─ Total Time:      2.393s (SDP: 2.380s, Rnd: 0.012s)

------------------------------------------------------------
